<a href="https://colab.research.google.com/github/Chimera418/dl-endsem-logical-ad/blob/main/notebooks/04_colab_phaseE_context.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Phase E — Global Context Modelling (`E18_Transformer` vs. `E19_Mamba`)
**Project:** 22AI304 / 22AIE304 End-Semester Capstone — Team 11  
**Protocol:** Controlled head-to-head comparison of **Self-Attention (`E18_Transformer`, `O(N^2 D)`)** vs. **2D Cross-Scan Selective SSM (`E19_Mamba`, `O(N D d_state)`)** on the exact same `16×16 = 256` spatial token grid inside `EnhancedPDN_S` (`200 epochs`, `AdamW cosine`, `patience=10`, native BF16).

**Prior Benchmarks:** `E01_PUAD_200ep` (`89.64%` / `84.07%`) | `E15_CrissCross` (`90.74%` / `85.33%`) | `E16_FPN` (`90.33%` / `87.84%`).

In [1]:
#@title 1. Verify Colab GPU, RAM, Disk & Install Dependencies
!pip install -q timm thop pyyaml scikit-learn pandas
import torch, psutil, shutil, sys
print('=== COLAB SESSION AUDIT (PHASE E) ===')
print(f'Python        : {sys.version.split()[0]}')
print(f'PyTorch       : {torch.__version__} | CUDA: {torch.version.cuda}')
if torch.cuda.is_available():
    p = torch.cuda.get_device_properties(0)
    native_bf16 = p.major >= 8
    print(f'GPU Model     : {p.name} (CC {p.major}.{p.minor}) | VRAM: {p.total_memory/1024**3:.2f} GiB | Native BF16: {native_bf16}')
else:
    raise RuntimeError('No GPU attached! Runtime -> Change runtime type -> L4 / G4 / A100 GPU.')
ram = psutil.virtual_memory()
disk = shutil.disk_usage('/content')
print(f'System RAM    : {ram.available/1e9:.1f} GB free / {ram.total/1e9:.1f} GB total')
print(f'Local SSD     : {disk.free/1e9:.1f} GB free / {disk.total/1e9:.1f} GB total')

=== COLAB SESSION AUDIT (PHASE E) ===
Python        : 3.13.16
PyTorch       : 2.14.1+cu130 | CUDA: 13.0
GPU Model     : NVIDIA L4 (CC 8.9) | VRAM: 22.03 GiB | Native BF16: True
System RAM    : 54.7 GB free / 56.9 GB total
Local SSD     : 253.0 GB free / 253.1 GB total


In [2]:
#@title 2. Mount Drive, Pull Latest Repo, Unpack Data & Run Section 3 Audit
from google.colab import drive
drive.mount('/content/drive')

import os
REPO_URL = 'https://github.com/Chimera418/dl-endsem-logical-ad.git'
DRIVE_ROOT = '/content/drive/MyDrive/DL_Endsem_Team11'
DRIVE_RUNS = f'{DRIVE_ROOT}/runs'
os.makedirs(DRIVE_RUNS, exist_ok=True)

if not os.path.exists('/content/Endsem/.git'):
    !git clone {REPO_URL} /content/Endsem
else:
    !git -C /content/Endsem pull --ff-only
%cd /content/Endsem

if not os.path.exists('/content/data/mvtec_loco_anomaly_detection/breakfast_box'):
    if os.path.exists('/content/data/breakfast_box'):
        os.environ['ENDSEM_LOCO_ROOT'] = '/content/data'
    else:
        os.makedirs('/content/data/mvtec_loco_anomaly_detection', exist_ok=True)
        !tar -xf "{DRIVE_ROOT}/data/mvtec_loco_anomaly_detection.tar.xz" -C /content/data/mvtec_loco_anomaly_detection/
        os.environ['ENDSEM_LOCO_ROOT'] = '/content/data/mvtec_loco_anomaly_detection'
else:
    os.environ['ENDSEM_LOCO_ROOT'] = '/content/data/mvtec_loco_anomaly_detection'

if not os.path.exists('/content/data/imagenet_penalty'):
    os.makedirs('/content/data/imagenet_penalty', exist_ok=True)
    !unzip -q "{DRIVE_ROOT}/data/imagenet_penalty_10k.zip" -d /content/data/imagenet_penalty

os.environ['ENDSEM_IMAGENET_ROOT'] = '/content/data/imagenet_penalty'
os.environ['ENDSEM_TEACHER_PATH'] = f'{DRIVE_ROOT}/weights/teacher_pdn_s.pt'

!python scripts/verify_env.py

Mounted at /content/drive
Cloning into '/content/Endsem'...
remote: Enumerating objects: 192, done.
remote: Counting objects: 100% (192/192), done.
remote: Compressing objects: 100% (132/132), done.
remote: Total 192 (delta 108), reused 130 (delta 56), pack-reused 0 (from 0)
Receiving objects: 100% (192/192), 187.93 KiB | 11.05 MiB/s, done.
Resolving deltas: 100% (108/108), done.
/content/Endsem
SECTION 3 — ENVIRONMENT & DATASET VERIFICATION AUDIT
1. CUDA Available    : True
2. GPU Model         : NVIDIA L4 (CC 8.9, BF16=True)
3. GPU Memory        : 22.03 GiB (22563 MiB)
4. PyTorch / CUDA    : PyTorch 2.14.1+cu130 | CUDA 13.0 | cuDNN 92800
   Python / Packages : Python 3.13.16 | timm=1.0.31 | thop=0.1.1 | mamba_ssm=None
5. System RAM        : 50.13 GB free / 52.96 GB total (5.3% used)
6. Disk Space        : 223.53 GB free / 235.68 GB total (/content/Endsem)
------------------------------------------------------------------------------
7. Dataset & Weights Accessibility:
   MVTec LOCO A

In [ ]:
#@title 3. Run Phase E (E18_Transformer & E19_Mamba) — Sequential for Clean Solo Latency
!python scripts/run_experiment.py --config configs/E18_transformer.yaml --mode train --drive-sync-dir "$DRIVE_RUNS"
!python scripts/run_experiment.py --config configs/E19_mamba.yaml --mode train --drive-sync-dir "$DRIVE_RUNS"

RUN: E18_Transformer | mode=train | device=cuda | seed=42
Dataset audit OK: train=1778 | val=305 | test_good=575 | test_logical=561 | structural_excluded=432
  [drive-restore] Restored best_checkpoint.pth from /content/drive/MyDrive/DL_Endsem_Team11/runs/E18_Transformer
  [drive-restore] Restored train_resume.pth from /content/drive/MyDrive/DL_Endsem_Team11/runs/E18_Transformer
Loading pooled train/good and validation/good (img_size=256, cache_in_ram=True)...
  Penalty corpus ready: 10,000 images (batch_size=8)
  Computing streaming teacher channel normalization stats on pooled train/good...
  [resume] Restored epoch 148 (iter 33,004, best val_loss 0.8640)
  Starting training: epochs=200, batch_size=8, amp=torch.bfloat16, trainable_params=7,664,129, frozen_params=2,694,144
  Epoch 149/200 | train=0.7636 | val=0.8647 | lr=5.6e-05 | 56.7s (patience 1/10)
  Epoch 150/200 | train=0.7620 | val=0.8569 | lr=5.4e-05 | 44.0s <-- BEST
  Epoch 151/200 | train=0.7607 | val=0.8699 | lr=5.2e-05 | 44

In [ ]:
#@title 4. Aggregate All Experiments (E01–E19 Master Table)
!python scripts/aggregate_results.py --dirs experiments "$DRIVE_RUNS" --out-dir "$DRIVE_RUNS/phaseE_summary"